# Big Mart Sales Prediction — Random Forest Regressor

### Objective
- Predict `Item_Outlet_Sales`.
- Use **Random Forest Regressor**.
- Follow a complete supervised regression workflow.
- Understand the data, visualize important patterns, build the model, evaluate it, and derive business recommendations.

### Dataset
- Source: KaggleHub
- Dataset: `yasserh/bigmartsalesdataset`
- File: `bigmart.csv`
- The dataset is loaded directly from KaggleHub, so no manual file path is required.


## Step 1 — Import Libraries and Connect to KaggleHub

### Why
- Import the tools required for data analysis, visualization, and machine learning.
- Connect directly to KaggleHub to make the notebook reproducible.

### How
- Download/access the Kaggle dataset using its dataset ID.
- Automatically locate `bigmart.csv`.


In [ ]:
# Install KaggleHub in the current Jupyter environment
%pip install -q kagglehub

# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

# Machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Connect directly to KaggleHub
path = kagglehub.dataset_download("yasserh/bigmartsalesdataset")

# Load the CSV
df = pd.read_csv(os.path.join(path, "bigmart.csv"))

print("KaggleHub connection successful")
print("Dataset shape:", df.shape)
print("Dataset path:", path)

display(df.head())


## Step 2 — Understand the Dataset

### Why
- Understand the number of observations and variables.
- Identify numerical and categorical data.
- Check the basic statistical characteristics.

### How
- Use `shape`, `info()` and `describe()`.


In [ ]:
# Dataset dimensions
print("Rows and columns:", df.shape)

# Column names and data types
print("\nColumn information:")
df.info()

# Statistical summary
print("\nStatistical summary:")
display(df.describe(include="all").T)


## Step 3 — Check Missing Values

### Why
- Missing values can cause problems during model training.
- We need to know which columns require treatment.

### How
- Count missing values in every column.
- Display only columns that contain missing values.
- Use a bar chart for easy identification.


In [ ]:
# Count missing values
missing = df.isnull().sum().sort_values(ascending=False)

print("Missing values:")
display(missing[missing > 0])

# Visualize missing values
missing_plot = missing[missing > 0]

plt.figure(figsize=(8, 4))
sns.barplot(x=missing_plot.values, y=missing_plot.index)
plt.title("Missing Values by Column")
plt.xlabel("Number of Missing Values")
plt.ylabel("Column")
plt.show()


## Step 4 — Clean the Data

### Why
- Numerical and categorical variables need different missing-value treatments.
- Cleaning makes the dataset ready for analysis and modeling.

### How
- `Item_Weight` → fill missing values with the median.
- `Outlet_Size` → fill using the most common value within `Outlet_Type`.
- Remove duplicate rows if present.
- Verify that missing values have been handled.


In [ ]:
# Fill numerical missing values with median
df["Item_Weight"] = df["Item_Weight"].fillna(df["Item_Weight"].median())

# Fill categorical missing values using Outlet_Type group mode
df["Outlet_Size"] = df["Outlet_Size"].fillna(
    df.groupby("Outlet_Type")["Outlet_Size"].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    )
)

# Final fallback for any remaining missing Outlet_Size values
df["Outlet_Size"] = df["Outlet_Size"].fillna(df["Outlet_Size"].mode().iloc[0])

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("Missing values after cleaning:")
display(df.isnull().sum())

print("Shape after duplicate removal:", df.shape)


## Step 5 — Analyze the Target Variable

### Why
- The target is `Item_Outlet_Sales`.
- Its distribution shows how sales values are spread across the dataset.
- This helps identify concentration and possible skewness.

### How
- Use a histogram with a density curve.


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["Item_Outlet_Sales"], kde=True)
plt.title("Distribution of Item Outlet Sales")
plt.xlabel("Item Outlet Sales")
plt.ylabel("Frequency")
plt.show()


## Step 6 — Compare Outlet Type with Sales

### Why
- Different outlet formats can have different sales performance.
- This comparison helps identify outlet types with higher or lower historical average sales.

### How
- Group the data by `Outlet_Type`.
- Calculate average `Item_Outlet_Sales`.
- Visualize the averages with a bar chart.

### Insight
- The highest and lowest average outlet types can be identified directly from the table.


In [ ]:
# Calculate average sales by outlet type
outlet_sales = (
    df.groupby("Outlet_Type")["Item_Outlet_Sales"]
    .mean()
    .sort_values(ascending=False)
)

display(outlet_sales.to_frame("Average Sales"))

# Visual comparison
plt.figure(figsize=(8, 4))
sns.barplot(
    data=df,
    x="Outlet_Type",
    y="Item_Outlet_Sales",
    estimator="mean",
    errorbar=None
)
plt.title("Average Sales by Outlet Type")
plt.xlabel("Outlet Type")
plt.ylabel("Average Sales")
plt.xticks(rotation=20)
plt.show()


## Step 7 — Compare Outlet Size with Sales

### Why
- Outlet size may be associated with selling capacity and customer traffic.
- Comparing average sales helps identify historical differences between outlet sizes.

### How
- Group by `Outlet_Size`.
- Calculate mean sales.
- Display the comparison as a bar chart.


In [ ]:
# Average sales by outlet size
size_sales = (
    df.groupby("Outlet_Size")["Item_Outlet_Sales"]
    .mean()
    .sort_values(ascending=False)
)

display(size_sales.to_frame("Average Sales"))

plt.figure(figsize=(7, 4))
sns.barplot(
    data=df,
    x="Outlet_Size",
    y="Item_Outlet_Sales",
    estimator="mean",
    errorbar=None
)
plt.title("Average Sales by Outlet Size")
plt.xlabel("Outlet Size")
plt.ylabel("Average Sales")
plt.show()


## Step 8 — Compare Item Type with Sales

### Why
- `Item_Type` represents different product categories.
- Comparing average sales helps identify categories with relatively higher or lower historical sales.

### How
- Group by `Item_Type`.
- Calculate average sales.
- Plot all categories for a clear comparison.


In [ ]:
# Average sales by item type
item_sales = (
    df.groupby("Item_Type")["Item_Outlet_Sales"]
    .mean()
    .sort_values(ascending=False)
)

display(item_sales.to_frame("Average Sales"))

plt.figure(figsize=(9, 6))
sns.barplot(x=item_sales.values, y=item_sales.index)
plt.title("Average Sales by Item Type")
plt.xlabel("Average Sales")
plt.ylabel("Item Type")
plt.show()


## Step 9 — Compare Item MRP with Sales

### Why
- `Item_MRP` and `Item_Outlet_Sales` are numerical variables.
- A scatter plot can show whether higher MRP values are associated with different sales levels.

### How
- Put MRP on the x-axis.
- Put sales on the y-axis.
- Each point represents an observation.

### Insight
- A visible pattern indicates a useful predictive relationship, but it does not prove causation.


In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(
    data=df,
    x="Item_MRP",
    y="Item_Outlet_Sales",
    alpha=0.4
)
plt.title("Item MRP vs Item Outlet Sales")
plt.xlabel("Item MRP")
plt.ylabel("Item Outlet Sales")
plt.show()


## Step 10 — Define Features and Target

### Why
- The model needs input variables and one output variable.
- The target must not be included in the features because that would cause data leakage.

### How
- `X` = all input features.
- `y` = `Item_Outlet_Sales`.
- Separate categorical and numerical columns for preprocessing.


In [ ]:
TARGET = "Item_Outlet_Sales"

# Features
X = df.drop(columns=TARGET)

# Target
y = df[TARGET]

# Separate column types
categorical = X.select_dtypes(include="object").columns.tolist()
numerical = X.select_dtypes(exclude="object").columns.tolist()

print("Target:", TARGET)
print("\nCategorical columns:")
print(categorical)

print("\nNumerical columns:")
print(numerical)

print("\nX shape:", X.shape)
print("y shape:", y.shape)


## Step 11 — Split Data into Training and Testing Sets

### Why
- The model should be evaluated on data it did not see during training.
- This gives a more realistic estimate of model performance.

### How
- 80% → training data.
- 20% → testing data.
- `random_state=42` makes the split reproducible.

### Kaggle Test Dataset
- Kaggle's competition `test.csv` normally has no `Item_Outlet_Sales`.
- The target is hidden because Kaggle expects us to predict it.
- Therefore, local evaluation uses a labelled 80/20 split.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))


## Step 12 — Encode Categorical Variables

### Why
- Machine-learning models require numerical input.
- Big Mart contains categorical columns such as outlet type and item type.
- One-Hot Encoding converts categories into numerical indicator columns.

### How
- Categorical columns → One-Hot Encoding.
- Numerical columns → kept as numerical values.
- `handle_unknown="ignore"` prevents errors if an unseen category appears.


In [ ]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", "passthrough", numerical)
])

print("Preprocessing pipeline created successfully.")


## Step 13 — Train the Random Forest Regressor

### Why
- This is a regression problem because sales are continuous numerical values.
- Random Forest can learn non-linear relationships and interactions.
- It combines many decision trees to produce a more robust prediction.

### Main settings
- `n_estimators=200` → 200 trees.
- `random_state=42` → reproducible results.
- `n_jobs=-1` → use available CPU cores.
- `max_features="sqrt"` → use a subset of features at each split.


In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        max_features="sqrt"
    ))
])

# Train the model
model.fit(X_train, y_train)

# Predict unseen test data
y_pred = model.predict(X_test)

print("Random Forest model trained successfully.")


## Step 14 — Evaluate the Model

### Why
- Regression models should not be evaluated using classification accuracy.
- We use MAE, RMSE and R².

### Metrics
- **MAE:** average absolute prediction error. Lower is better.
- **RMSE:** penalizes large errors more strongly. Lower is better.
- **R²:** proportion of target variation explained by the model. Higher is better.

### Result
- The values below are the actual evaluation results for this notebook's 20% test split.


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MODEL PERFORMANCE")
print("-" * 40)
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")


## Step 15 — Actual vs Predicted Sales

### Why
- Numerical metrics give summary scores.
- This graph visually checks prediction quality.

### How
- Actual sales → x-axis.
- Predicted sales → y-axis.
- Dashed diagonal → perfect prediction line.

### Insight
- Points close to the diagonal represent smaller prediction errors.
- Points farther away represent larger errors.


In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    x=y_test,
    y=y_pred,
    alpha=0.45
)

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

plt.plot([low, high], [low, high], linestyle="--")

plt.title("Actual vs Predicted Sales")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.show()


## Step 16 — Feature Importance

### Why
- Random Forest provides feature-importance values.
- They help explain which variables were most useful for prediction.

### How
- Extract transformed feature names.
- Match them with Random Forest importance scores.
- Display the top 15 features.

### Insight
- Higher importance means stronger predictive contribution to this model.
- Importance does not prove that a feature causes higher sales.


In [ ]:
pre = model.named_steps["preprocessor"]
rf = model.named_steps["regressor"]

feature_names = pre.get_feature_names_out()

importance = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("Top 15 features:")
display(importance.head(15).to_frame("Importance"))

plt.figure(figsize=(9, 6))
sns.barplot(
    x=importance.head(15).values,
    y=importance.head(15).index
)
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


## Step 17 — Business Insights

### Why
- Machine-learning results should be translated into understandable business findings.

### How
- Compare average sales by outlet type and item type.
- Check numerical correlations with sales.
- Combine these observations with feature importance.

### Key interpretation
- Higher average sales identifies stronger historical-performing groups.
- Feature importance identifies variables that were useful to the model.
- These are predictive observations, not proof of causation.


In [ ]:
print("Highest average outlet type:", outlet_sales.index[0])
print("Lowest average outlet type :", outlet_sales.index[-1])

print("\nTop 5 item types by average sales:")
display(item_sales.head(5).to_frame("Average Sales"))

print("\nNumerical correlations with sales:")
correlation = (
    df.select_dtypes(include=np.number)
    .corr()["Item_Outlet_Sales"]
    .sort_values(ascending=False)
)

display(correlation.to_frame("Correlation"))


## Step 18 — Business Recommendations

### Based on the analysis
- Use predicted sales to support inventory planning.
- Study high-performing outlet types and their operating characteristics.
- Allocate inventory according to predicted demand.
- Use product and MRP patterns when planning product mix and pricing.
- Monitor low-performing products and outlets to reduce overstocking.
- Retrain the model periodically using newer sales data.
- Use predictions as decision-support information, not guaranteed future sales.


In [ ]:
print("BUSINESS RECOMMENDATIONS")
print("-" * 40)

recommendations = [
    "1. Use sales predictions to support inventory planning.",
    "2. Study high-performing outlet types and their operating practices.",
    "3. Allocate stock according to predicted demand.",
    "4. Use product and MRP patterns when planning product mix and pricing.",
    "5. Monitor low-performing products and outlets to reduce overstocking.",
    "6. Retrain the model when newer sales data becomes available.",
    "7. Use predictions as decision-support information, not guaranteed future sales."
]

for recommendation in recommendations:
    print(recommendation)


## Step 19 — Final Conclusion

### Project outcome
- The dataset was connected directly through KaggleHub.
- Data quality was checked and missing values were handled.
- EDA was used to understand sales patterns.
- Categorical variables were encoded.
- A Random Forest Regressor was trained.
- MAE, RMSE and R² were used for regression evaluation.
- Actual-vs-predicted and feature-importance visualizations were used to interpret the model.
- Business insights and recommendations were derived from the analysis.

### Final model metrics
- **MAE:** shown by the model evaluation cell.
- **RMSE:** shown by the model evaluation cell.
- **R²:** shown by the model evaluation cell.


In [ ]:
print("FINAL MODEL RESULTS")
print("-" * 40)
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")
print("\nBig Mart Sales Prediction project completed successfully.")
